In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [ ]:
# Чтение очищенного датасета
df = pd.read_excel("../data/vk_posts_clean.xlsx")
print(f"Размер: {df.shape}")
df.head(5)
df.info()

In [ ]:
#Предобработка текста
def preprocess_text(text):
    if pd.isnull(text):
        return ""
    return text.lower()

df["Текст поста"] = df["Текст поста"].apply(preprocess_text)

#Словарь ключевых фраз по категориям
category_keywords = {
    "Матч / Результаты": ["счёт", "победа", "проигрыш", "проигрываем", "ничья", "гол", "матч", "результат", "состав", "преодалела"],
    "Анонс / Расписание": ["анонс", "начало", "во сколько", "где смотреть", "дата", "стадион"],
    "Праздники / Поздравления": ["поздравляем", "с праздником", "рождество", "новый год", "пасха", "праздник"],
    "Юмор / Мемы": ["мем", "шутка", "угар", "прикол"],
    "Интерактив / Опрос": ["опрос", "проголосуй", "как думаете", "ваше мнение", "комментарии"]
}

In [ ]:
#Функция классификации поста
def classify_post(text):
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if kw in text:
                return category
    return "Другое"

df["Категория"] = df["Текст поста"].apply(classify_post)


In [ ]:
# Сохранение датасета с категориями (пригодится для BI и анализа)
df.to_excel("../data/vk_posts_with_categories.xlsx", index=False)

In [ ]:
# Работаем с исходным df
df_model = df.copy()

#Фичи на основе даты
df_model['Дата публикации'] = pd.to_datetime(df_model['Дата публикации'])
df_model['Час'] = df_model['Дата публикации'].dt.hour
df_model['День недели'] = df_model['Дата публикации'].dt.dayofweek

#Фичи на основе текста
df_model['Длина текста'] = df_model['Текст поста'].apply(len)
df_model['Количество эмодзи'] = df_model['Текст поста'].apply(lambda x: len(re.findall(r'[^\w\s,]', x)))

In [ ]:
#Кодирование категориальных признаков
df_model = pd.get_dummies(df_model, columns=['Категория'], drop_first=True)

#Целевые переменные
target_views = df_model['Просмотры']
target_likes = df_model['Лайки']

#Признаки
feature_columns = [
    'Комментарии', 'Час', 'День недели', 'Длина текста', 'Количество эмодзи'
] + [col for col in df_model.columns if col.startswith('Категория_')]

X = df_model[feature_columns]

#Делим выборку
X_train, X_test, y_train_views, y_test_views = train_test_split(X, target_views, test_size=0.2, random_state=42)
_, _, y_train_likes, y_test_likes = train_test_split(X, target_likes, test_size=0.2, random_state=42)

#Обучение
model_views = RandomForestRegressor(random_state=42)
model_likes = RandomForestRegressor(random_state=42)

model_views.fit(X_train, y_train_views)
model_likes.fit(X_train, y_train_likes)

pred_views = model_views.predict(X_test)
pred_likes = model_likes.predict(X_test)


In [ ]:
#Оценка эффективности
metrics = {
    "MAE (Views)": mean_absolute_error(y_test_views, pred_views),
    "R2 (Views)": r2_score(y_test_views, pred_views),
    "MAE (Likes)": mean_absolute_error(y_test_likes, pred_likes),
    "R2 (Likes)": r2_score(y_test_likes, pred_likes),
}

#Матрица корреляции
correlation_matrix = df_model[feature_columns + ['Просмотры', 'Лайки']].corr()

#Построение графика корреляции
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Матрица корреляции признаков и целевых переменных")
plt.tight_layout()
plt.show()

metrics

In [ ]:
# Сохранение моделей и вспомогательных объектов
import os
os.makedirs("models", exist_ok=True)
joblib.dump(model_views, "models/model_views.pkl")
joblib.dump(model_likes, "models/model_likes.pkl")
joblib.dump(feature_columns, "models/features.pkl")
joblib.dump(category_keywords, "models/keywords.pkl")